<a href="https://colab.research.google.com/github/ehas1/Statistical-Bias-in-ML/blob/main/OAIP_Skeleton_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports & Models

In [ ]:
# imports
import pandas as pd
import requests
from io import StringIO, BytesIO
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, confusion_matrix, log_loss
from graphviz import Digraph
from joblib import dump, load
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential, save_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import os
import datetime
from google.colab import drive

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# Setup storage and models directory
def setup_storage():
    """Set up storage and return the models directory path."""
    # First try to detect if we're in Colab
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if IN_COLAB:
        try:
            from google.colab import drive
            print("Google Colab environment detected. Mounting Google Drive...")
            drive.mount('/content/gdrive')
            models_dir = "/content/gdrive/MyDrive/COMPAS_models"
        except Exception as e:
            print(f"Error mounting Google Drive: {str(e)}")
            print("Falling back to local directory...")
            models_dir = os.path.join(os.getcwd(), 'COMPAS_models')
    else:
        print("Local environment detected. Using local directory.")
        models_dir = os.path.join(os.path.expanduser('~'), 'COMPAS_models')

    # Create the directory if it doesn't exist
    try:
        os.makedirs(models_dir, exist_ok=True)
        print(f"Models will be saved to: {models_dir}")
    except Exception as e:
        print(f"Error creating directory: {str(e)}")
        # Fallback to current directory if home directory is not accessible
        models_dir = os.path.join(os.getcwd(), 'COMPAS_models')
        os.makedirs(models_dir, exist_ok=True)
        print(f"Falling back to current directory: {models_dir}")

    return models_dir

models_dir = setup_storage()
# Updated save_model_with_metadata function for Google Drive integration
def save_model_with_metadata(model, model_type: str, metadata: dict, file_extension: str = 'joblib'):
    """Save a model and its metadata directly to Google Drive.

    Args:
        model: The trained model to save
        model_type: Type of model ('decision_tree', 'xgboost', or 'neural_network')
        metadata: Dictionary containing model metadata
        file_extension: File extension for the model file (default: 'joblib')

    Returns:
        tuple: (model_path, metadata_path)
    """
    # Clean up old model and metadata files of the same type
    def safe_remove(filepath):
        try:
            if os.path.exists(filepath):
                os.remove(filepath)
                return True
            return False
        except Exception as e:
            print(f'Error removing file {filepath}: {str(e)}')
            return False

    files_removed = 0
    for file in os.listdir(models_dir):
        # Match both model and metadata files for the specific model type
        if file.startswith(f'{model_type}_model_'):
            old_file_path = os.path.join(models_dir, file)
            if safe_remove(old_file_path):
                files_removed += 1
                print(f'Removed old file: {file}')

    if files_removed > 0:
        print(f'\nCleanup complete: Removed {files_removed} old file(s)')
    else:
        print('\nNo old files found to clean up')

    # Generate timestamp for unique filenames
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

    # Create file paths with model type in the name
    # For XGBoost, always use .json extension
    if model_type == 'xgboost':
        model_filename = f'{model_type}_model_{timestamp}.json'
    else:
        model_filename = f'{model_type}_model_{timestamp}.{file_extension}'
    metadata_filename = f'{model_type}_model_{timestamp}_metadata.json'
    model_path = os.path.join(models_dir, model_filename)
    metadata_path = os.path.join(models_dir, metadata_filename)

    # Add timestamp and model type to metadata
    metadata['timestamp'] = timestamp
    metadata['model_type'] = model_type

    try:
        # Save model based on type
        if model_type == 'neural_network':
            model.save(model_path)
        elif model_type == 'xgboost':
            model.save_model(model_path)  # Will save as JSON due to .json extension
        else:  # decision_tree
            dump(model, model_path)

        # Save metadata
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=4)

        print(f'\nModel saved as: {model_path}')
        print(f'Model metadata saved as: {metadata_path}')

    except Exception as e:
        print(f"Error saving model: {str(e)}")

    return model_path, metadata_path


#Functions

In [ ]:
# define functions



def create_model_with_params(input_dim, params):
    """Create a neural network model with specified hyperparameters."""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu', kernel_regularizer=l2(0.02)),
        BatchNormalization(),
        Dropout(0.4),

        # Second block
        Dense(64, activation='relu', kernel_regularizer=l2(0.02)),
        BatchNormalization(),
        Dropout(0.3),

        # Third block
        Dense(32, activation='relu', kernel_regularizer=l2(0.02)),
        BatchNormalization(),
        Dropout(0.2),

        # Output layer
        Dense(1, activation='sigmoid', kernel_regularizer=l2(0.02))
    ])

    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer,
                 loss='binary_crossentropy',
                 metrics=['accuracy',
                         tf.keras.metrics.AUC(name='auc'),
                         tf.keras.metrics.Precision(name='precision'),
                         tf.keras.metrics.Recall(name='recall')])
    return model

def grid_search_nn(train_data, param_grid, n_fold=5, random_state=42):
    """Perform grid search with cross-validation for neural network hyperparameters.

    Args:
        train_data: List containing [X_train, y_train]
        param_grid: Dictionary of hyperparameter combinations to try
        n_fold: Number of cross-validation folds
        random_state: Random seed for reproducibility

    Returns:
        dict: Best parameters and their corresponding metrics
    """
    X_train, y_train = train_data
    kf = KFold(n_splits=n_fold, shuffle=True, random_state=random_state)

    best_val_score = 0
    best_params = None
    best_histories = []
    all_results = {}

    for params in param_grid:
        print(f"\nTrying parameters: {params}")
        fold_histories = []
        fold_val_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
            print(f"Fold {fold + 1}/{n_fold}")

            # Split data
            X_train_fold = X_train.iloc[train_idx]
            y_train_fold = y_train.iloc[train_idx]
            X_val_fold = X_train.iloc[val_idx]
            y_val_fold = y_train.iloc[val_idx]

            # Create and train model
            model = create_model_with_params(X_train.shape[1], params)

            # Callbacks
            early_stopping = EarlyStopping(
                monitor='val_loss',
                patience=5,
                restore_best_weights=True
            )
            reduce_lr = ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.2,
                patience=3,
                min_lr=1e-6
            )

            history = model.fit(
                X_train_fold, y_train_fold,
                epochs=15,
                batch_size=params['batch_size'],
                validation_data=(X_val_fold, y_val_fold),
                callbacks=[early_stopping, reduce_lr],
                verbose=0
            )

            # Store results
            val_score = max(history.history['val_accuracy'])
            fold_val_scores.append(val_score)
            fold_histories.append(history.history)

            # Clear session to free memory
            tf.keras.backend.clear_session()

        # Calculate mean validation score for this parameter set
        mean_val_score = np.mean(fold_val_scores)
        std_val_score = np.std(fold_val_scores)

        print(f"Mean validation accuracy: {mean_val_score:.4f} (±{std_val_score:.4f})")

        # Store results for this parameter combination
        all_results[str(params)] = {
            'mean_val_score': mean_val_score,
            'std_val_score': std_val_score,
            'histories': fold_histories
        }

        # Update best parameters if necessary
        if mean_val_score > best_val_score:
            best_val_score = mean_val_score
            best_params = params
            best_histories = fold_histories

    return {
        'best_params': best_params,
        'best_val_score': best_val_score,
        'best_histories': best_histories,
        'all_results': all_results
    }

def plot_training_history(histories, title='Training History'):
    """Plot detailed training history with confidence intervals.

    Args:
        histories: List of training histories from different folds
        title: Plot title
    """
    metrics = ['accuracy', 'loss', 'auc', 'precision', 'recall']
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, metric in enumerate(metrics):
        if metric not in histories[0]:
            continue

        val_metric = f'val_{metric}'

        # Collect metric values across folds
        train_values = []
        val_values = []
        max_epochs = max(len(h[metric]) for h in histories)

        for history in histories:
            # Pad shorter histories with NaN
            train_metric = history[metric]
            val_metric_values = history[val_metric]

            train_padded = train_metric + [np.nan] * (max_epochs - len(train_metric))
            val_padded = val_metric_values + [np.nan] * (max_epochs - len(val_metric_values))

            train_values.append(train_padded)
            val_values.append(val_padded)

        train_values = np.array(train_values)
        val_values = np.array(val_values)

        # Calculate mean and std
        train_mean = np.nanmean(train_values, axis=0)
        train_std = np.nanstd(train_values, axis=0)
        val_mean = np.nanmean(val_values, axis=0)
        val_std = np.nanstd(val_values, axis=0)

        epochs = range(1, max_epochs + 1)

        # Plot
        axes[idx].plot(epochs, train_mean, label='Train', color='blue')
        axes[idx].fill_between(epochs,
                             train_mean - train_std,
                             train_mean + train_std,
                             alpha=0.1,
                             color='blue')

        axes[idx].plot(epochs, val_mean, label='Validation', color='red')
        axes[idx].fill_between(epochs,
                             val_mean - val_std,
                             val_mean + val_std,
                             alpha=0.1,
                             color='red')

        axes[idx].set_title(f'{metric.capitalize()}')
        axes[idx].set_xlabel('Epoch')
        axes[idx].set_ylabel(metric.capitalize())
        axes[idx].legend()
        axes[idx].grid(True)

    # Remove empty subplot if any
    if len(metrics) < len(axes):
        fig.delaxes(axes[-1])

    plt.suptitle(title)
    plt.tight_layout()
    return fig

def load_data(url="https://raw.githubusercontent.com/propublica/compas-analysis/refs/heads/master/cox-violent-parsed.csv") -> pd.DataFrame:
    """Download COMPAS data set from Github and load it directly into memory.

    Args:
        url (str): URL to the COMPAS dataset CSV file

    Returns:
        pd.DataFrame: Loaded COMPAS dataset
    """
    response = requests.get(url)
    data = pd.read_csv(StringIO(response.text))
    return data


def preprocess_data(data: pd.DataFrame, random_state: int = 42) -> tuple:
    """Preprocess COMPAS data set with feature selection, encoding, and normalization.

    Args:
        data: Raw COMPAS dataset
        random_state: Random seed for reproducibility

    Returns:
        tuple: (train_data, test_data) where each is [X_scaled, y]
    """

    # Remove duplicate records
    initial_size = len(data)
    data = data.drop_duplicates()
    duplicates_removed = initial_size - len(data)
    print(f"\nPreprocessing Summary:")
    print(f"Initial number of records: {initial_size}")
    print(f"Number of duplicates removed: {duplicates_removed}")
    print(f"Final number of records: {len(data)}")
    # Remove invalid recidivism records and select features
    data = data[data['is_recid'] != -1]
    features = ["sex", "age", "race", "juv_fel_count", "juv_misd_count",
                "juv_other_count", "priors_count", "c_charge_degree"]
    X = data[features].copy()  # Create an explicit copy
    y = data['is_recid']

    # Categorize charge degree
    X.loc[:, 'c_charge_degree'] = X['c_charge_degree'].fillna('Other')
    X.loc[:, 'c_charge_degree'] = X['c_charge_degree'].apply(
        lambda x: 'Felony' if str(x).startswith('F') else
                 'Misdemeanor' if str(x).startswith('M') else 'Other'
    )

    # One-hot encode categorical features
    X_encoded = pd.get_dummies(X, columns=['sex', 'race', 'c_charge_degree'])

    # Normalize numerical features
    numerical_features = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']
    for col in numerical_features:
        min_val = X_encoded[col].min()
        max_val = X_encoded[col].max()
        if max_val > min_val:
            X_encoded[col] = (X_encoded[col] - min_val) / (max_val - min_val)

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2,
                                                        random_state=random_state)

    return [X_train, y_train], [X_test, y_test]


def train_model(train_data: list, model_type: str = 'decision_tree', params: dict = None):
    """Train either a decision tree or XGBoost model on the training data.

    Args:
        train_data: List containing [X_train, y_train]
        model_type: Either 'decision_tree' or 'xgboost'
        params: Dictionary of model parameters
    """
    X_train, y_train = train_data
    if params is None:
        params = {}

    if model_type == 'decision_tree':
        print(params)
        model = DecisionTreeClassifier(**params)
    elif model_type == 'xgboost':
        model = xgb.XGBClassifier(**params)
    else:
        raise ValueError("model_type must be either 'decision_tree' or 'xgboost'")

    model.fit(X_train, y_train)
    return model

def train_with_crossvalidation(train_data: list, model_type: str = 'decision_tree',
  n_fold: int = 5, params: dict = None, shuffle=True, random_state=42) -> dict:
  """Split training data into folds, train and evaluate model on each fold."""

  X_train, y_train = train_data
  kf = KFold(n_splits=n_fold, shuffle=shuffle, random_state=random_state)
  models = ["" for i in range(n_fold)]
  cv_evals = ["" for i in range(n_fold)]

  for i, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    # Split data using indices
    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train model on this fold
    model = train_model([X_train_fold, y_train_fold], model_type=model_type, params=params)
    models[i] = model

    # Evaluate on train and validation sets
    cv_evals[i] = {
        "train_evals": test_model(model, [X_train_fold, y_train_fold]),
        "validation_evals": test_model(model, [X_val_fold, y_val_fold])
    }

  return cv_evals

def optimize_depth(train_data: list, model_type: str = 'decision_tree',
                  min_td: int = 2, max_td: int = 6,
                  n_fold: int = 5, params: dict = None,
                  overfitting_penalty: float = 0.3) -> tuple:
  """Perform crossvalidated training pipeline for several values of tree_depth.

  Args:
      train_data: Training data as [X, y]
      model_type: Type of model to train ('decision_tree' or 'xgboost')
      min_td: Minimum tree depth to try
      max_td: Maximum tree depth to try
      n_fold: Number of cross-validation folds
      params: Additional model parameters
      overfitting_penalty: Weight for penalizing overfitting (default: 0.3)
  """
  optimization_evals = {}
  depth_analysis = {}

  for tree_depth in range(min_td, max_td+1):
    if params is None:
        params = {}
    params['max_depth'] = tree_depth
    cv_evals = train_with_crossvalidation(train_data,
      model_type=model_type, n_fold=n_fold, params=params)
    optimization_evals[tree_depth] = cv_evals

    # Calculate metrics for this depth
    val_acc = np.mean([fold['validation_evals']['accuracy'] for fold in cv_evals])
    val_std = np.std([fold['validation_evals']['accuracy'] for fold in cv_evals])
    train_acc = np.mean([fold['train_evals']['accuracy'] for fold in cv_evals])
    overfitting_gap = train_acc - val_acc

    depth_analysis[tree_depth] = {
        'train_acc': train_acc,
        'val_acc': val_acc,
        'val_std': val_std,
        'overfitting_gap': overfitting_gap
    }

  # Select best depth based on validation accuracy and overfitting gap
  best_depth = max(depth_analysis.keys(),
                  key=lambda d: depth_analysis[d]['val_acc'] - 0.5 * depth_analysis[d]['overfitting_gap'])
  best_depth = 4

  # Calculate overall accuracy as mean of all validation accuracies across all folds
  overall_accuracies = []
  for fold in optimization_evals[best_depth]:
      overall_accuracies.append(fold['validation_evals']['accuracy'])
  overall_accuracy = np.mean(overall_accuracies)

  print(f"\nBest depth: {best_depth}")
  print(f"Overall Accuracy: {overall_accuracy:.1%}")
  print(f"Validation Accuracy: {depth_analysis[best_depth]['val_acc']:.7f}")
  print(f"Overfitting Gap: {depth_analysis[best_depth]['overfitting_gap']:.7f}")

  return optimization_evals, best_depth, depth_analysis

def tree_viz(model: DecisionTreeClassifier) -> Digraph:
    """Create a detailed visualization of a decision tree using graphviz.

    Args:
        model: Trained DecisionTreeClassifier model

    Returns:
        Digraph: Graphical representation of the decision tree with detailed node information
    """
    # Create directed graph
    fig = Digraph(comment='Decision Tree Visualization')
    fig.attr(rankdir='TB')  # Top to bottom layout

    # Set visualization style
    fig.attr('node', shape='box', style='rounded,filled', fillcolor='lightblue')
    fig.attr('edge', color='gray50')
    fig.attr(bgcolor='white')

    # Get tree structure
    n_nodes = model.tree_.node_count
    children_left = model.tree_.children_left
    children_right = model.tree_.children_right
    feature = model.tree_.feature
    threshold = model.tree_.threshold
    value = model.tree_.value

    # Create nodes with simplified information
    for node_id in range(n_nodes):
        if children_left[node_id] == -1:  # Leaf node
            prob = value[node_id][0, 1] / value[node_id].sum()
            color = 'lightpink' if prob >= 0.5 else 'lightgreen'
            label = f'Prediction: {"Yes" if prob >= 0.5 else "No"}\nRecidivism Prob: {prob:.1%}'
            fig.node(str(node_id), label, fillcolor=color)
        else:  # Decision node
            feat_name = model.feature_names_in_[feature[node_id]]
            split_value = threshold[node_id]

            # Format the split value based on feature type
            if feat_name in ['sex_Female', 'sex_Male'] or feat_name.startswith('race_') or feat_name.startswith('c_charge_degree_'):
                # For one-hot encoded features, show as is (0 or 1)
                label = f'{feat_name}\n= {int(split_value)}'
            else:
                # For numerical features, show as integers if they're counts
                if feat_name in ['juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']:
                    label = f'{feat_name}\n≤ {int(round(split_value))}'
                else:  # For age, keep one decimal place
                    label = f'{feat_name}\n≤ {int(round(split_value * 100))}'

            fig.node(str(node_id), label)

            # Add edges to children with Yes/No labels
            fig.edge(str(node_id), str(children_left[node_id]), 'Yes')
            fig.edge(str(node_id), str(children_right[node_id]), 'No')

    # Set graph size and font
    fig.attr(size='12,12')
    fig.attr(fontsize='10')

    return fig


def test_model(model: DecisionTreeClassifier, data: list) -> dict:
    """Evaluate model performance with comprehensive metrics.

    Args:
        model: Trained classifier model
        data: List containing [X_test, y_test]

    Returns:
        dict: Performance metrics including accuracy, precision, recall, F1,
              ROC-AUC, confusion matrix, and loss
    """
    X_test, y_test = data
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]  # Probability of positive class

    # Calculate confusion matrix values
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    evals = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),  # Same as TPR
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'loss': log_loss(y_test, y_prob),
        'tpr': tp / (tp + fn),  # True Positive Rate
        'fpr': fp / (fp + tn),  # False Positive Rate
        'tnr': tn / (tn + fp),  # True Negative Rate
        'fnr': fn / (fn + tp),  # False Negative Rate
    }

    return evals


def plot_evals(evals: dict) -> plt.figure:
    """Create a compact visualization of model evaluation metrics.

    Args:
        evals: Dictionary containing model evaluation metrics

    Returns:
        plt.figure: Figure containing the visualizations
    """
    # Set style
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (12, 4)

    # Create figure with subplots
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3)

    # Plot 1: Accuracy metrics
    metrics1 = ['accuracy', 'precision', 'recall', 'f1']
    values1 = [evals[m] for m in metrics1]
    sns.barplot(x=metrics1, y=values1, ax=ax1)
    ax1.set_title('Accuracy Metrics')
    ax1.set_ylim(0, 1)

    # Plot 2: Error rates
    metrics2 = ['tpr', 'fpr', 'tnr', 'fnr']
    values2 = [evals[m] for m in metrics2]
    sns.barplot(x=metrics2, y=values2, ax=ax2)
    ax2.set_title('Error Rates')
    ax2.set_ylim(0, 1)

    # Plot 3: ROC-AUC and Loss
    metrics3 = ['roc_auc', 'loss']
    values3 = [evals[m] for m in metrics3]
    sns.barplot(x=metrics3, y=values3, ax=ax3)
    ax3.set_title('ROC-AUC & Loss')

    # Adjust layout
    plt.tight_layout()

    return fig

def plot_cv_evals(cv_evals: list) -> plt.figure:
    """Visualize cross-validation results with mean performance and confidence intervals.

    Args:
        cv_evals: List containing evaluation metrics for each fold

    Returns:
        plt.figure: Figure showing mean performance with min-max envelope
    """
    # Set style
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (10, 6)

    # Extract metrics from all folds
    train_metrics = []
    val_metrics = []
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']

    for fold in cv_evals:
        train_metrics.append([fold['train_evals'][metric] for metric in metrics_to_plot])
        val_metrics.append([fold['validation_evals'][metric] for metric in metrics_to_plot])

    # Convert to numpy arrays for easier computation
    train_metrics = np.array(train_metrics)
    val_metrics = np.array(val_metrics)

    # Create figure
    cv_fig, ax = plt.subplots()
    x = np.arange(len(metrics_to_plot))

    # Plot training metrics
    ax.plot(x, train_metrics.mean(axis=0), 'b-', label='Train (mean)', linewidth=2)
    ax.fill_between(x, train_metrics.min(axis=0), train_metrics.max(axis=0),
                   alpha=0.2, color='blue', label='Train (min-max)')

    # Plot validation metrics
    ax.plot(x, val_metrics.mean(axis=0), 'r-', label='Validation (mean)', linewidth=2)
    ax.fill_between(x, val_metrics.min(axis=0), val_metrics.max(axis=0),
                   alpha=0.2, color='red', label='Validation (min-max)')

    # Customize plot
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.set_title('Cross-validation Performance Metrics')
    ax.legend(loc='lower right')
    plt.tight_layout()

    return cv_fig


def plot_optimization_evals(evals: dict) -> plt.figure:
    """Create plots that show training and validation accuracy as a function of tree depth.

    Args:
        evals: Dictionary containing evaluation metrics for each tree depth

    Returns:
        plt.figure: Figure showing accuracy vs tree depth for both training and validation

    EH: Change so plot shows different depths
    """
    # Set style
    sns.set_style('whitegrid')
    plt.rcParams['figure.figsize'] = (10, 6)

    # Initialize lists to store metrics
    tree_depths = []
    train_accuracies = []
    train_accuracies_min = []
    train_accuracies_max = []
    val_accuracies = []
    val_accuracies_min = []
    val_accuracies_max = []

    # Extract metrics for each tree depth
    for depth, cv_evals in evals.items():
        tree_depths.append(depth)

        # Get training accuracies across folds
        train_acc = [fold['train_evals']['accuracy'] for fold in cv_evals]
        train_accuracies.append(np.mean(train_acc))
        train_accuracies_min.append(np.min(train_acc))
        train_accuracies_max.append(np.max(train_acc))

        # Get validation accuracies across folds
        val_acc = [fold['validation_evals']['accuracy'] for fold in cv_evals]
        val_accuracies.append(np.mean(val_acc))
        val_accuracies_min.append(np.min(val_acc))
        val_accuracies_max.append(np.max(val_acc))

    # Create figure
    fig, ax = plt.subplots()

    # Plot training metrics
    ax.plot(tree_depths, train_accuracies, 'b-', label='Train (mean)', linewidth=2)
    ax.fill_between(tree_depths, train_accuracies_min, train_accuracies_max,
                   alpha=0.2, color='blue', label='Train (min-max)')

    # Plot validation metrics
    ax.plot(tree_depths, val_accuracies, 'r-', label='Validation (mean)', linewidth=2)
    ax.fill_between(tree_depths, val_accuracies_min, val_accuracies_max,
                   alpha=0.2, color='red', label='Validation (min-max)')

    # Customize plot
    ax.set_xlabel('Tree Depth')
    ax.set_ylabel('Accuracy')
    ax.set_title('Model Performance vs Tree Depth')
    ax.legend(loc='lower right')
    ax.grid(True)

    # Set axis limits
    ax.set_ylim(0, 1)
    ax.set_xticks(tree_depths)

    plt.tight_layout()
    print(train_acc)
    print(val_acc)

    return fig

def create_model(input_dim):
    """Create a neural network model with the specified architecture."""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu', kernel_regularizer=l2(0.02)),
        BatchNormalization(),
        Dropout(0.4),

        # Second block
        Dense(64, activation='relu', kernel_regularizer=l2(0.02)),
        BatchNormalization(),
        Dropout(0.3),

        # Third block
        Dense(32, activation='relu', kernel_regularizer=l2(0.02)),
        BatchNormalization(),
        Dropout(0.2),

        # Output layer
        Dense(1, activation='sigmoid', kernel_regularizer=l2(0.02))
    ])

    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer,
                 loss='binary_crossentropy',
                 metrics=['accuracy', tf.keras.metrics.AUC()])
    return model

def train_neural_network(train_data, validation_data=None, epochs=15):
    """Train a neural network with early stopping and learning rate reduction."""
    X_train, y_train = train_data

    # Create callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6
    )

    callbacks = [early_stopping, reduce_lr]

    # Create and train model
    model = create_model(input_dim=X_train.shape[1])

    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=32,
        validation_split=0.2 if validation_data is None else 0,
        validation_data=validation_data,
        callbacks=callbacks,
        verbose=1
    )

    # Print training summary
    print("\nTraining Summary:")
    print(f"Total epochs trained: {len(history.history['loss'])}")
    print(f"Best validation loss: {min(history.history['val_loss']):.4f}")
    print(f"Final training accuracy: {history.history['accuracy'][-1]:.2%}")
    if 'val_accuracy' in history.history:
        print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.2%}")
    if 'auc' in history.history:
        print(f"Final training AUC: {history.history['auc'][-1]:.4f}")

    return model, history

def train_with_crossvalidation_nn(train_data, n_fold=5, random_state=42):
    """Train neural network with k-fold cross-validation."""
    X_train, y_train = train_data
    kf = KFold(n_splits=n_fold, shuffle=True, random_state=random_state)
    cv_evals = []

    # Lists to store metrics across folds
    accuracies = []
    balanced_accuracies = []
    roc_aucs = []
    f1_scores = []
    mccs = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        print(f"\nFold {fold + 1}/{n_fold}")

        # Split data
        X_train_fold = X_train.iloc[train_idx]
        y_train_fold = y_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]
        y_val_fold = y_train.iloc[val_idx]

        # Train model
        model, history = train_neural_network(
            [X_train_fold, y_train_fold],
            validation_data=(X_val_fold, y_val_fold)
        )

        # Evaluate
        train_evals = test_model_nn(model, [X_train_fold, y_train_fold], print_metrics=False)
        val_evals = test_model_nn(model, [X_val_fold, y_val_fold], print_metrics=False)

        # Store validation metrics
        accuracies.append(val_evals['accuracy'])
        balanced_accuracies.append(val_evals['balanced_accuracy'])
        roc_aucs.append(val_evals['roc_auc'])
        f1_scores.append(val_evals['f1'])
        mccs.append(val_evals['mcc'])

        cv_evals.append({
            "train_evals": train_evals,
            "validation_evals": val_evals,
            "history": history.history
        })

        # Clear session to free memory
        tf.keras.backend.clear_session()

    # Print cross-validation summary
    print("\nCross-validation Summary:")
    print(f"Mean Accuracy: {np.mean(accuracies):.2%} (±{np.std(accuracies):.2%})")
    print(f"Mean Balanced Accuracy: {np.mean(balanced_accuracies):.2%} (±{np.std(balanced_accuracies):.2%})")
    print(f"Mean ROC AUC: {np.mean(roc_aucs):.4f} (±{np.std(roc_aucs):.4f})")
    print(f"Mean F1 Score: {np.mean(f1_scores):.4f} (±{np.std(f1_scores):.4f})")
    print(f"Mean Matthews Correlation Coefficient: {np.mean(mccs):.4f} (±{np.std(mccs):.4f})")

    return cv_evals

def setup_model_directory():
    """Create and return the path to the models directory."""
    models_dir = "/content/drive/MyDrive/COMPAS_models/"
    os.makedirs(models_dir, exist_ok=True)
    return models_dir

def save_model_with_metadata(model, model_type: str, metadata: dict, file_extension: str = 'joblib'):
    """Save a model and its metadata to the models directory.

    Args:
        model: The trained model to save
        model_type: Type of model ('decision_tree', 'xgboost', or 'neural_network')
        metadata: Dictionary containing model metadata
        file_extension: File extension for the model file (default: 'joblib')
    """
    # Setup directory and timestamp
    models_dir = setup_model_directory()
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

    # Clean up old model files of the same type
    for file in os.listdir(models_dir):
        if file.startswith(f'{model_type}_model_'):
            old_file_path = os.path.join(models_dir, file)
            try:
                os.remove(old_file_path)
                print(f'Removed old model file: {file}')
            except Exception as e:
                print(f'Error removing old file {file}: {str(e)}')

    # Create file paths with model type in the name
    model_path = os.path.join(models_dir, f'{model_type}_model_{timestamp}.{file_extension}')
    metadata_path = os.path.join(models_dir, f'{model_type}_model_{timestamp}_metadata.json')

    # Add timestamp and model type to metadata
    metadata['timestamp'] = timestamp
    metadata['model_type'] = model_type

    # Save model based on type
    if model_type == 'neural_network':
        model.save(model_path)
    elif model_type == 'xgboost':
        model.save_model(model_path)
    else:  # decision_tree
        dump(model, model_path)

    # Save metadata
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=4)

    print(f'\nModel saved as: {model_path}')
    print(f'Model metadata saved as: {metadata_path}')


def test_model_nn(model, data, print_metrics=True):
    """Evaluate neural network performance with comprehensive metrics."""
    X_test, y_test = data
    y_pred = (model.predict(X_test) > 0.5).astype(int)
    y_prob = model.predict(X_test)

    # Calculate confusion matrix values
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    loss = log_loss(y_test, y_prob)
    tpr = tp / (tp + fn)  # True Positive Rate (Sensitivity)
    fpr = fp / (fp + tn)  # False Positive Rate
    tnr = tn / (tn + fp)  # True Negative Rate (Specificity)
    fnr = fn / (fn + tp)  # False Negative Rate

    # Calculate additional metrics
    balanced_accuracy = (tpr + tnr) / 2
    mcc = (tp * tn - fp * fn) / np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn) + 1e-8)  # Matthews Correlation Coefficient

    evals = {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'mcc': mcc,
        'loss': loss,
        'tpr': tpr,
        'fpr': fp / (fp + tn), # Corrected FPR calculation
        'tnr': tn / (tn + fp),  # True Negative Rate (Specificity)
        'fnr': fn / (fn + tp),  # False Negative Rate
    }

    if print_metrics:
        print("\nModel Evaluation Metrics:")
        print(f"Overall Accuracy: {accuracy:.2%}")
        print(f"Balanced Accuracy: {balanced_accuracy:.2%}")
        print(f"Precision: {precision:.2%}")
        print(f"Recall (Sensitivity): {recall:.2%}")
        print(f"Specificity: {tnr:.2%}")
        print(f"F1 Score: {f1:.4f}")
        print(f"ROC AUC: {roc_auc:.4f}")
        print(f"Matthews Correlation Coefficient: {mcc:.4f}")
        print(f"Log Loss: {loss:.4f}")
        print("\nConfusion Matrix Metrics:")
        print(f"True Positives: {tp}")
        print(f"True Negatives: {tn}")
        print(f"False Positives: {fp}")
        print(f"False Negatives: {fn}")

    return evals


def xgb_tree_viz_styled(tree_dump_json: str, feature_names: list, base_score: float = 0.5) -> Digraph:
    """Create a stylized visualization of a single XGBoost tree dump using graphviz,
    mimicking the appearance and node info of the Decision Tree visualization.

    Args:
        tree_dump_json: JSON string representing the XGBoost tree structure
        feature_names: List of feature names
        base_score: The base score of the XGBoost model (used for probability calculation)

    Returns:
        Digraph: Graphical representation of the XGBoost tree with detailed node information and styling
    """
    # Create directed graph
    fig = Digraph(comment='XGBoost Tree Visualization')
    fig.attr(rankdir='TB')  # Top to bottom layout

    # Set visualization style to match Decision Tree but with larger size
    # Enhanced visualization style
    fig.attr(dpi='300')  # Increase DPI for better quality
    fig.attr('node',
        shape='box',
        style='rounded,filled',
        fillcolor='lightblue',
        width='1.5',  # Increase node width
        height='1.2',  # Increase node height
        margin='0.2,0.2',  # Add margin around text
        fontname='Arial',  # Use a clean font
        fontsize='10')  # Smaller font for better fit
    fig.attr('edge',
        color='gray50',
        penwidth='2.0',  # Thicker edges
        fontsize='9',    # Smaller font for edge labels
        fontname='Arial')
    fig.attr(bgcolor='white')
    fig.attr(size='30,30')  # Larger overall size
    fig.attr(ratio='compress')  # Better use of space

    tree_dict = json.loads(tree_dump_json)

    def calculate_probability(leaf_value, base_score):
        """Calculate probability from raw XGBoost leaf value (log-odds)."""
        log_odds = leaf_value + base_score
        log_odds = float(log_odds)
        log_odds = np.clip(log_odds, -10, 10)
        probability = 1 / (1 + np.exp(-log_odds))
        return probability

    def add_nodes_edges(node_dict, graph, feature_names, base_score):
        if not isinstance(node_dict, dict):
            return

        node_id = str(node_dict['nodeid'])

        if 'leaf' in node_dict:  # Leaf node
            value = node_dict['leaf']
            # Show raw contribution value in a more concise format
            raw_value = float(value)
            # Color based on whether the contribution is positive or negative
            color = 'lightpink' if raw_value > 0 else 'lightgreen'
            # More compact label with sign and value
            sign = "+" if raw_value > 0 else ""  # minus sign will be included in the number if negative
            label = f'{sign}{raw_value:.2f}'
            graph.node(node_id, label, fillcolor=color)
        else:  # Decision node
            split_info = node_dict.get('split')
            split_condition = node_dict.get('split_condition')
            yes_child_id = node_dict.get('yes')
            no_child_id = node_dict.get('no')

            # Handle special node types and splits
            missing_direction = node_dict.get('missing')
            default_direction = node_dict.get('default_left', None)

            # Get quantile information if available
            quantile_info = node_dict.get('quantile_info', {})
            has_quantiles = 'quantile_boundaries' in quantile_info

            if split_info is None or split_condition is None:
                # Handle truly incomplete nodes
                label = f"Node ID: {node_id}\n(Incomplete node)"
                graph.node(node_id, label, fillcolor='orange')
            else:
                # Format the split condition
                feat_name = feature_names[split_info] if isinstance(split_info, int) else split_info

                if has_quantiles:
                    # Handle quantile splits
                    boundaries = quantile_info['quantile_boundaries']
                    label = f"{feat_name}\nQuantiles: {boundaries}"
                else:
                    # Handle regular splits with missing value direction - more concise format
                    split_value = split_condition
                    # Shorten feature names for better display
                    short_name = feat_name.replace('_count', '').replace('charge_', '')
                    label = f"{short_name}\n≤ {split_value:.2f}"  # Use fewer decimal places

                    # Add missing/default info more compactly
                    if missing_direction is not None or default_direction is not None:
                        extra_info = []
                        if missing_direction is not None:
                            extra_info.append(f"M→{'L' if missing_direction == yes_child_id else 'R'}")
                        if default_direction is not None:
                            extra_info.append(f"D→{'L' if default_direction else 'R'}")
                        if extra_info:
                            label += f"\n({', '.join(extra_info)})"

                graph.node(node_id, label, fillcolor='lightblue')

            # Add edges to children with appropriate labels
            if 'children' in node_dict:
                for child in node_dict['children']:
                    if isinstance(child, dict) and 'nodeid' in child:
                        child_id = str(child['nodeid'])
                        # Determine edge label based on child type
                        if child_id == str(yes_child_id):
                            edge_label = 'Yes'
                        elif child_id == str(no_child_id):
                            edge_label = 'No'
                        elif missing_direction is not None and child_id == str(missing_direction):
                            edge_label = 'Missing'
                        else:
                            edge_label = '?'

                        graph.edge(node_id, child_id, edge_label)
                        add_nodes_edges(child, graph, feature_names, base_score)

            # Determine feature name
            if isinstance(split_info, int):
                feat_name = feature_names[split_info] if split_info < len(feature_names) else f"Unknown Feature (Index {split_info})"
            else:
                feat_name = split_info

            split_value = split_condition

            # Format split value exactly like tree_viz with type checking
            if split_value is None:
                label = f'{feat_name}\n(split value missing)'
            elif feat_name in ['sex_Female', 'sex_Male'] or feat_name.startswith('race_') or feat_name.startswith('c_charge_degree_'):
                try:
                    label = f'{feat_name}\n= {int(float(split_value))}'
                except (ValueError, TypeError):
                    label = f'{feat_name}\n= {split_value}'
            else:
                try:
                    if feat_name in ['juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']:
                        label = f'{feat_name}\n≤ {int(round(float(split_value)))}'
                    elif feat_name == 'age':
                        label = f'{feat_name}\n≤ {int(round(float(split_value) * 100))}'
                    else:
                        label = f'{feat_name}\n≤ {float(split_value):.3f}'
                except (ValueError, TypeError):
                    label = f'{feat_name}\n≤ {split_value}'

            graph.node(node_id, label)

            # Add edges with Yes/No labels exactly like tree_viz
            yes_child_dicts = [item for item in node_dict.get('children', []) if isinstance(item, dict) and str(item.get('nodeid')) == str(yes_child_id)]
            no_child_dicts = [item for item in node_dict.get('children', []) if isinstance(item, dict) and str(item.get('nodeid')) == str(no_child_id)]

            yes_child_dict = yes_child_dicts[0] if yes_child_dicts else None
            no_child_dict = no_child_dicts[0] if no_child_dicts else None

            if yes_child_dict:
                graph.edge(node_id, str(yes_child_dict['nodeid']), 'Yes')
                add_nodes_edges(yes_child_dict, graph, feature_names, base_score)

            if no_child_dict:
                graph.edge(node_id, str(no_child_dict['nodeid']), 'No')
                add_nodes_edges(no_child_dict, graph, feature_names, base_score)

    # Find and process root node
    root_node = None
    parsed_dump = json.loads(tree_dump_json)
    if isinstance(parsed_dump, list) and len(parsed_dump) > 0:
        for node in parsed_dump:
            if isinstance(node, dict) and node.get('nodeid') == 0:
                root_node = node
                break
        if root_node is None and isinstance(parsed_dump[0], dict):
            root_node = parsed_dump[0]
    elif isinstance(parsed_dump, dict) and parsed_dump.get('nodeid') == 0:
        root_node = parsed_dump
    elif isinstance(parsed_dump, dict):
        root_node = parsed_dump

    if root_node is None:
        print("Error: Could not find a valid root node in tree dump.")
        return fig

    add_nodes_edges(root_node, fig, feature_names, base_score)

    return fig

In [ ]:
# DATA

data = load_data()
train_data, test_data = preprocess_data(data)

# Decision Tree

In [ ]:
# DECISION TREES - RACIAL AND SEX BIAS ANALYSIS

# Set initial parameters for racial bias tree
racial_bias_params = {
    'criterion': 'gini',
    'min_samples_split': 10,
    'min_samples_leaf': 4,
    'random_state': 42,
    'ccp_alpha': 0.002,
    'max_features': 0.7,
    'class_weight': {0: 1, 1: 1.2},
    'splitter': 'best'
}

# Set initial parameters for sex bias tree
sex_bias_params = {
    'criterion': 'entropy',
    'min_samples_split': 20,
    'min_samples_leaf': 8,
    'random_state': 42,
    'ccp_alpha': 0.0075,
    'max_features': 'sqrt',
    'class_weight': 'balanced'
}

min_td, max_td = 2, 8  # Extended max depth to allow for more complex trees

print("\n=== Training Racial Bias Decision Tree ===")
optimization_evals_racial, best_depth_racial, depth_analysis_racial = optimize_depth(
    train_data,
    model_type='decision_tree',
    min_td=min_td,
    max_td=max_td,
    n_fold=5,
    params=racial_bias_params,
    overfitting_penalty=0.3
)

# Plot optimization results for racial bias tree
opt_fig_racial = plot_optimization_evals(optimization_evals_racial)
plt.title("Racial Bias Tree - Optimization Results")
plt.show()

# Train final racial bias model with best depth
racial_bias_params['max_depth'] = best_depth_racial
model_dt_racial = train_model(train_data, model_type='decision_tree', params=racial_bias_params)
test_evals_racial = test_model(model_dt_racial, test_data)
eval_fig_racial = plot_evals(test_evals_racial)
plt.title("Racial Bias Tree - Performance Metrics")
plt.show()

print("\nGenerating racial bias decision tree visualization...")
model_dt_racial.feature_names_in_ = train_data[0].columns.tolist()
tree_graph_racial = tree_viz(model_dt_racial)
display(tree_graph_racial)

# Print feature importance scores for racial bias tree
print("\nFeature Importance Scores (Racial Bias Tree):")
feature_importance_racial = pd.DataFrame({
    'feature': train_data[0].columns,
    'importance': model_dt_racial.feature_importances_
})
feature_importance_racial = feature_importance_racial.sort_values('importance', ascending=False)
print(feature_importance_racial.to_string(index=False))

# Save racial bias model and metadata
metadata_racial = {
    'model_purpose': 'racial_bias_analysis',
    'best_depth': best_depth_racial,
    'validation_accuracy': depth_analysis_racial[best_depth_racial]['val_acc'],
    'overfitting_gap': depth_analysis_racial[best_depth_racial]['overfitting_gap'],
    'parameters': racial_bias_params
}
save_model_with_metadata(model_dt_racial, 'racial_bias_decision_tree', metadata_racial)

print("\n=== Training Sex Bias Decision Tree ===")
# Train and evaluate sex bias tree
optimization_evals_sex, best_depth_sex, depth_analysis_sex = optimize_depth(
    train_data,
    model_type='decision_tree',
    min_td=min_td,
    max_td=max_td,
    n_fold=5,
    params=sex_bias_params,
    overfitting_penalty=0.3
)

# Plot optimization results for sex bias tree
opt_fig_sex = plot_optimization_evals(optimization_evals_sex)
plt.title("Sex Bias Tree - Optimization Results")
plt.show()

# Train final sex bias model with best depth
sex_bias_params['max_depth'] = best_depth_sex
model_dt_sex = train_model(train_data, model_type='decision_tree', params=sex_bias_params)
test_evals_sex = test_model(model_dt_sex, test_data)
eval_fig_sex = plot_evals(test_evals_sex)
plt.title("Sex Bias Tree - Performance Metrics")
plt.show()

print("\nGenerating sex bias decision tree visualization...")
model_dt_sex.feature_names_in_ = train_data[0].columns.tolist()
tree_graph_sex = tree_viz(model_dt_sex)
display(tree_graph_sex)

# Print feature importance scores for sex bias tree
print("\nFeature Importance Scores (Sex Bias Tree):")
feature_importance_sex = pd.DataFrame({
    'feature': train_data[0].columns,
    'importance': model_dt_sex.feature_importances_
})
feature_importance_sex = feature_importance_sex.sort_values('importance', ascending=False)
print(feature_importance_sex.to_string(index=False))

# Save sex bias model and metadata
metadata_sex = {
    'model_purpose': 'sex_bias_analysis',
    'best_depth': best_depth_sex,
    'validation_accuracy': depth_analysis_sex[best_depth_sex]['val_acc'],
    'overfitting_gap': depth_analysis_sex[best_depth_sex]['overfitting_gap'],
    'parameters': sex_bias_params
}
save_model_with_metadata(model_dt_sex, 'sex_bias_decision_tree', metadata_sex)

#XGBoost

In [ ]:
# XGBOOST

# Set initial parameters for XGBoost
params = {
    'learning_rate': 0.05,
    'max_depth': 3,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'random_state': 42,
    'n_estimators': 9,
}

# Run hyperparameter optimization and get evaluation results
min_td, max_td = 2, 6
optimization_evals, best_depth, depth_analysis = optimize_depth(train_data, model_type='xgboost',
                                                              min_td=min_td, max_td=max_td,
                                                              n_fold=5, params=params)

# Train final model with best depth and evaluate final model performance
params['max_depth'] = best_depth
model_xgb = train_model(train_data, model_type='xgboost', params=params)
test_evals = test_model(model_xgb, test_data)

# Plot final model evaluation metrics
eval_fig = plot_evals(test_evals)
plt.show()

# Visualize all 9 XGBoost trees using the styled visualization
print("\nGenerating XGBoost trees visualization...")

# Get feature names from training data
feature_names = train_data[0].columns.tolist()

# Get tree dumps in JSON format
trees = model_xgb.get_booster().get_dump(dump_format='json')

# Create a figure with a 3x3 grid
n_trees = len(trees)
n_cols = min(3, n_trees)
n_rows = (n_trees + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, 25))
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # Reduce spacing between subplots
axes = axes.ravel()

# Visualize each tree using xgb_tree_viz_styled
for idx, tree_dump in enumerate(trees):
    print(f"Visualizing Tree {idx + 1}")

    # Create styled tree visualization
    tree_graph = xgb_tree_viz_styled(tree_dump, feature_names, base_score=0.5)

    # Convert to PNG and display in the grid
    png_data = tree_graph.pipe(format='png')
    img = plt.imread(BytesIO(png_data))

    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f'Tree {idx + 1}')

# Remove empty subplots if any
for i in range(n_trees, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()

# Save the combined visualization as a high-resolution PNG
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
save_path = f'xgboost_trees_{timestamp}.png'
plt.savefig(save_path, dpi=400, bbox_inches='tight', pad_inches=0.5)
print(f"\nXGBoost trees visualization saved as: {save_path}")

plt.show()

print("XGBoost tree visualization complete.")

# Also save individual trees as separate PNGs
print("\nSaving individual trees...")
for idx, tree_dump in enumerate(trees):
    # Create individual tree visualization
    tree_graph = xgb_tree_viz_styled(tree_dump, feature_names, base_score=0.5)

    # Save individual tree
    individual_save_path = f'xgboost_tree_{idx + 1}_{timestamp}.png'
    tree_graph.render(filename=f'tree_{idx + 1}', format='png', cleanup=True)
    os.rename(f'tree_{idx + 1}.png', individual_save_path)
    print(f"Tree {idx + 1} saved as: {individual_save_path}")

# Save model and metadata
metadata = {
    'best_depth': best_depth,
    'validation_accuracy': depth_analysis[best_depth]['val_acc'],
    'overfitting_gap': depth_analysis[best_depth]['overfitting_gap'],
    'parameters': params
}
save_model_with_metadata(model_xgb, 'xgboost', metadata)

# Neural Network

In [ ]:
# NEURAL NETWORK WITH HYPERPARAMETER TUNING AND CROSS-VALIDATION

# Define parameter grid for search
param_grid = [
    {
        'batch_size': 32,
        'learning_rate': 0.001,
    }
]

#print("Starting hyperparameter search with cross-validation...")
#search_results = grid_search_nn(train_data, param_grid)

# Train final model on full training set with best parameters
print("\nTraining final model on full training set with best parameters...")
final_model_2, history_2 = train_neural_network(train_data)
final_model = create_model_with_params(train_data[0].shape[1], 0)


# Convert data to tensors for faster training
X_train_values = train_data[0].values.astype('float32')
y_train_values = train_data[1].values.astype('float32')
X_train = tf.convert_to_tensor(X_train_values)
y_train = tf.convert_to_tensor(y_train_values)

# Create training dataset with shuffling and prefetching
full_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_size = int(0.8 * len(X_train))  # 80% for training

# Split into training and validation datasets
train_dataset = full_dataset.take(train_size).batch(32)
    #.prefetch(tf.data.AUTOTUNE)

val_dataset = full_dataset.skip(train_size).batch(32)
    #.prefetch(tf.data.AUTOTUNE)

# Callbacks for final training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6
    )
]

# Train on full training set with validation split
history = final_model.fit(
    X_train, y_train,
    epochs=15,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

# Evaluate on test set
print("\nEvaluating final model 2 on test set...")
test_evals = test_model_nn(final_model_2, test_data, print_metrics=True)
print("\nEvaluating final model on test set...")
test_evals = test_model_nn(final_model, test_data, print_metrics=True)


# Prepare metadata for saving
metrics = {
    'training_history': {
        'epochs_trained': len(history.history['loss']),
        'best_val_loss': min(history.history['val_loss']),
        'final_accuracy': float(history.history['accuracy'][-1]),
        'final_val_accuracy': float(history.history['val_accuracy'][-1])
    },
    'test_metrics': test_evals
}

# Save model and metadata
print("\nSaving final model and metadata...")
save_model_with_metadata(final_model, 'neural_network', metrics, 'keras')
